In [1]:
!pip install lightning -q
import os, tarfile
from pathlib import Path

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
import torchvision
from torchvision import transforms

SEED = 0
L.seed_everything(SEED)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 17.6 MB/s eta 0:00:0000:01


Seed set to 0


0

# Загрузка Датасета, Даталоадеры

In [2]:
class CIFARDataModule(L.LightningDataModule):
    def __init__(self, batch_size=128, val_size=5000, num_workers=2, seed=SEED):
        super().__init__()
        self.batch_size = batch_size
        self.val_size = val_size
        self.num_workers = num_workers
        self.seed = seed

        self.transform_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
        ])
        self.transform_test = transforms.Compose([transforms.ToTensor()])

    def _locate_data_root(self):
        work = Path("/kaggle/working/data") if Path("/kaggle/working").exists() else Path("./data")
        work.mkdir(parents=True, exist_ok=True)

        if Path("/kaggle/input").exists():
            for p in Path("/kaggle/input").rglob("cifar-10-batches-py"):
                if p.is_dir():
                    link = work / "cifar-10-batches-py"
                    if not link.exists():
                        os.symlink(p, link)
                    print("Найдены готовые батчи:", p)
                    return work, False

            for p in Path("/kaggle/input").rglob("cifar-10-python.tar.gz"):
                if not (work / "cifar-10-batches-py").exists():
                    print("Распаковываю", p)
                    with tarfile.open(p) as tar:
                        tar.extractall(work)
                return work, False

        print("Датасет не найден в /kaggle/input, буду скачивать (нужен Internet: On)")
        return work, True

    def prepare_data(self):
        self.data_root, self.need_download = self._locate_data_root()
        torchvision.datasets.CIFAR10(self.data_root, train=True, download=self.need_download)
        torchvision.datasets.CIFAR10(self.data_root, train=False, download=self.need_download)

    def setup(self, stage=None):
        full_train = torchvision.datasets.CIFAR10(self.data_root, train=True, transform=self.transform_train)
        val_source = torchvision.datasets.CIFAR10(self.data_root, train=True, transform=self.transform_test)
        self.test_set = torchvision.datasets.CIFAR10(self.data_root, train=False, transform=self.transform_test)

        g = torch.Generator().manual_seed(self.seed)
        n_train = len(full_train) - self.val_size
        train_idx, val_idx = random_split(range(len(full_train)), [n_train, self.val_size], generator=g)

        self.train_set = Subset(full_train, list(train_idx))
        self.val_set = Subset(val_source, list(val_idx))

    def train_dataloader(self):
        return DataLoader(
            self.train_set, batch_size=self.batch_size, shuffle=True,
            num_workers=self.num_workers, pin_memory=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_set, batch_size=256, shuffle=False,
            num_workers=self.num_workers, pin_memory=True
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_set, batch_size=256, shuffle=False,
            num_workers=self.num_workers, pin_memory=True
        )

cifar_dm = CIFARDataModule()

# Загрузка модели

Первая свёртка `3x3, stride 1` вместо `7x7, stride 2`, начальный `maxpool` убран. Без этого картинка 32x32 сжимается до 8x8 ещё до первого residual-блока, и последние слои работают с картой 1x1.

Атака должна работать в пространстве `[0, 1]`, иначе ограничение вроде eps = 8/255 теряет смысл — поэтому нормализация не в трансформах, а внутри модели (класс `Normalize`).

Отличия CIFAR-версии от `torchvision.models.resnet18`: `conv1` это `3x3, stride 1, padding 1`, и `maxpool` заменён на `Identity`. Размеры карт по стадиям: 32 → 32 → 16 → 8 → 4.

In [3]:
class Normalize(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        return (x - self.mean) / self.std


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * self.expansion),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out)


class ResNetCIFAR(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10,
                 mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)):
        super().__init__()
        self.normalize = Normalize(mean, std)
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(block, 64,  num_blocks[0], 1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], 2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], 2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], 2)
        self.linear = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.normalize(x)
        out = F.relu(self.bn1(self.conv1(out)))
        out = self.layer1(out); out = self.layer2(out)
        out = self.layer3(out); out = self.layer4(out)
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.linear(out)


class ResNetClassifier(L.LightningModule):
    def __init__(self, num_classes=10, lr=0.1, momentum=0.9, weight_decay=5e-4, max_epochs=30):
        super().__init__()
        self.save_hyperparameters()
        self.model = ResNetCIFAR(BasicBlock, [2, 2, 2, 2], num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, on_epoch=True)
        self.log("train_acc", acc, on_epoch=True)
        self.log("lr", self.trainer.optimizers[0].param_groups[0]["lr"], on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_loss", loss, on_epoch=True, sync_dist=True)
        self.log("val_acc", acc, on_epoch=True, sync_dist=True)

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(
            self.parameters(), lr=self.hparams.lr, momentum=self.hparams.momentum,
            weight_decay=self.hparams.weight_decay, nesterov=True
        )
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=self.hparams.lr,
            total_steps=self.trainer.estimated_stepping_batches
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},
        }


model = ResNetClassifier(num_classes=10, lr=0.1, momentum=0.9, weight_decay=5e-4, max_epochs=30)
n_params = sum(p.numel() for p in model.parameters())
print(f"Параметров: {n_params/1e6:.2f}M")

Параметров: 11.17M


# Обучение

In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_acc", mode="max", save_top_k=1, filename="resnet18-best"
)

trainer = L.Trainer(
    devices=1,
    max_epochs=30,
    accelerator="auto",
    precision="16-mixed",
    callbacks=[checkpoint_callback],
)
trainer.fit(model, datamodule=cifar_dm)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Датасет не найден в /kaggle/input, буду скачивать (нужен Internet: On)


100%|██████████| 170M/170M [00:23<00:00, 7.21MB/s] 
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type        ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ResNetCIFAR │ 11.2 M │ train │     0 │
└───┴───────┴─────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44.696                                                                     
Modules in train mode: 63                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

In [ ]:
best_model = ResNetClassifier.load_from_checkpoint(checkpoint_callback.best_model_path)
torch.save(best_model.cpu().state_dict(), 'resnet18_cifar10.pth')